# LexRAG: pomiar modeli na GPU (Google Colab)

Notatnik uruchamia LexRAG na karcie GPU Colaba (T4 16 GB wystarcza), buduje indeks ustawy o FUS, mierzy pięć modeli na złotym zbiorze 40 pytań i na końcu pozwala zadawać własne pytania.

**Zasada bezpieczeństwa:** do Colaba trafiają wyłącznie **akty publiczne** (ustawy, rozporządzenia) i złoty zbiór pytań. Instrukcje wewnętrzne, procedury i dokumenty instytucji **nigdy** nie są tu wgrywane.

**Przed startem:** w panelu „Pliki” (ikona folderu po lewej) wgraj `lexrag-colab.zip` do pamięci sesji, tak by leżał jako `/content/lexrag-colab.zip`. Potem „Środowisko wykonawcze → Uruchom wszystko”; komórki 6 i 7 są domyślnie wyłączone flagą, więc nic nie zatrzyma przebiegu. Czas: przygotowanie około 5 minut, każdy model 10 do 20 minut na T4. Wyniki częściowe pojawiają się po każdym modelu.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Ollama
Najpierw pakiet `zstd` (wymagany przez instalator, Colab go nie ma), potem instalator pobiera najnowszą Ollamę (≥ 0.40, bo Saluki ma architekturę `qwen35`). Serwer startuje w tle; log w `/content/ollama.log`.

In [ ]:
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import os, subprocess, time, urllib.request
env = dict(os.environ, OLLAMA_HOST="127.0.0.1:11434", OLLAMA_KEEP_ALIVE="30m")
ollama_proc = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(90):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/"); break
    except Exception:
        time.sleep(1)
!ollama --version

## 2. Kod LexRAG
Rozpakowanie paczki wgranej przez panel „Pliki”, instalacja bibliotek i 31 testów jednostkowych.

In [ ]:
import glob, os
ZIP = sorted(glob.glob("/content/lexrag-colab*.zip"), key=os.path.getmtime)[-1]
print("paczka:", ZIP)
!rm -rf /content/lexrag && mkdir -p /content/lexrag && unzip -q -o {ZIP} -d /content/lexrag
%cd /content/lexrag
!pip -q install -r requirements.txt -r requirements-ml.txt pymupdf
!python -m pytest -q

## 3. Ustawa i indeks
Tekst jednolity ustawy o FUS (Dz.U. 2024 poz. 1631) z API Sejmu. Embeddingi liczą się na GPU. Ostatnia komenda mierzy samo wyszukiwanie: recall@10 powinien wynieść 100 %.

In [ ]:
!python -m lexrag fetch DU/2024/1631
!python -m lexrag wgraj data/raw/DU_2024_1631.pdf --kod fus --akt "Ustawa o emeryturach i rentach z FUS" --obowiazuje-od 2024-11-07 --index data/index_fus
!python -m lexrag show fus:art24:ust1 --index data/index_fus
!python -m lexrag eval data/golden/fus.jsonl --index data/index_fus

## 4. Pomiar modeli
Modele małe z biblioteki Ollamy. Te same ustawienia co na laptopie: bez analizy pytania, domyślny budżet kontekstu, pełna weryfikacja. Po każdym modelu drukowane jest zestawienie częściowe.

In [ ]:
import os, re
MODELE = [
    "gemma4:e4b",
    "qwen3:4b",
    "qwen3:8b",
    "gemma4:12b",
]
os.environ["LEXRAG_SKIP_ANALYSIS"] = "1"
for m in MODELE:
    nazwa = re.sub(r"[^a-z0-9]+", "_", m.lower().split("/")[-1]).strip("_")
    os.environ["LEXRAG_LLM_MODEL"] = m
    print("=" * 30, m, "=" * 30, flush=True)
    !ollama pull {m}
    !python -m lexrag llm-test
    !python -m lexrag eval-odpowiedzi data/golden/fus.jsonl --index data/index_fus --bez-analizy --log data/eval_odpowiedzi_fus_{nazwa}.jsonl
    !ollama stop {m}
    !python scripts/porownaj_modele.py

## 4b. Saluki 27B (Qwen3.5-27B w 2 bitach)
Plik GGUF w repozytorium Hugging Face ma niestandardową nazwę (`IQ2-mix`), więc `ollama pull hf.co/...` go nie znajduje. Pobieramy plik (7,9 GB) i tworzymy model `saluki` z Modelfile. To ten model odpowiada na pytanie, czy duży model daje 99 %.

In [ ]:
!curl -L -s -o /content/saluki.gguf https://huggingface.co/ConwayResearch/Underdog-Saluki-27B-1.0/resolve/main/Underdog-Saluki-27B-1.0-IQ2-mix.gguf
!ls -la /content/saluki.gguf
!printf 'FROM /content/saluki.gguf\n' > /content/Modelfile.saluki && ollama create saluki -f /content/Modelfile.saluki && rm /content/saluki.gguf
import os
os.environ["LEXRAG_LLM_MODEL"] = "saluki"
os.environ["LEXRAG_SKIP_ANALYSIS"] = "1"
!python -m lexrag llm-test
!python -m lexrag eval-odpowiedzi data/golden/fus.jsonl --index data/index_fus --bez-analizy --log data/eval_odpowiedzi_fus_saluki_t4.jsonl
!ollama stop saluki
!python scripts/porownaj_modele.py

## 5. Zestawienie
Precyzja = poprawne / udzielone odpowiedzi (cel „99 %”); pokrycie = udzielone / wszystkie pytania. Plik `/content/wyniki_lexrag.zip` pobierz z panelu „Pliki” (prawy przycisk → Pobierz) i wrzuć do `data/` w repozytorium na laptopie.

In [ ]:
!python scripts/porownaj_modele.py
!zip -q -j /content/wyniki_lexrag.zip data/eval_odpowiedzi_fus_*.jsonl && ls -la /content/wyniki_lexrag.zip

## 6. Zadawaj pytania (ręcznie)
Ustaw model i uruchom tę komórkę. Indeks, embeddingi i reranker zostają w pamięci, więc kolejne pytania są szybkie. Pusta linia kończy. Przykłady: `od jakiego wieku przysługuje emerytura osobom urodzonym po 1948 roku`, `za jaki okres wstecz można żądać zwrotu nienależnie pobranych świadczeń`, `czy trzeba rozwiązać umowę o pracę żeby ZUS wypłacał emeryturę`.

In [ ]:
TRYB_INTERAKTYWNY = False      # zmień na True i uruchom komórkę, żeby pytać
MODEL = "gemma4:12b"           # albo "gemma4:e4b", "qwen3:4b"
if TRYB_INTERAKTYWNY:
    import os
    os.environ["LEXRAG_LLM_MODEL"] = MODEL
    os.environ["LEXRAG_SKIP_ANALYSIS"] = "1"
    from lexrag.index import Indeks
    from lexrag.generate import odpowiedz
    from lexrag.cli import wypisz_odpowiedz
    idx = Indeks.wczytaj("data/index_fus")
    print("Model:", MODEL, "| jednostek w indeksie:", len(idx))
    while True:
        pytanie = input("\nPytanie (puste = koniec): ").strip()
        if not pytanie:
            break
        w = odpowiedz(idx, pytanie, analiza=False)
        wypisz_odpowiedz(w, idx, pokaz_kontekst=False)
        print("czasy [s]:", w.czasy)

## 7. (opcjonalnie) Zapis na Dysku Google
Colab kasuje `/content` po zamknięciu sesji. Ta komórka kopiuje indeks i logi do folderu `LexRAG` na Dysku (wymaga zgody w oknie Google; dlatego domyślnie wyłączona).

In [ ]:
ZAPISZ_NA_DYSKU = False      # zmień na True i uruchom komórkę, żeby skopiować wyniki na Dysk Google
if ZAPISZ_NA_DYSKU:
    from google.colab import drive
    drive.mount("/content/drive")
    get_ipython().system("mkdir -p /content/drive/MyDrive/LexRAG && cp -r data/index_fus data/eval_odpowiedzi_fus_*.jsonl /content/drive/MyDrive/LexRAG/ ; ls -la /content/drive/MyDrive/LexRAG")